## Homework 3: Symbolic Music Generation Using Markov Chains

**Before starting the homework:**

Please run `pip install miditok` to install the [MiDiTok](https://github.com/Natooz/MidiTok) package, which simplifies MIDI file processing by making note and beat extraction more straightforward.

You’re also welcome to experiment with other MIDI processing libraries such as [mido](https://github.com/mido/mido), [pretty_midi](https://github.com/craffel/pretty-midi) and [miditoolkit](https://github.com/YatingMusic/miditoolkit). However, with these libraries, you’ll need to handle MIDI quantization yourself, for example, converting note-on/note-off events into beat positions and durations.

In [1]:
# run this command to install MiDiTok
#! pip install miditok

In [2]:
# import required packages
import random
from glob import glob
from collections import defaultdict

import numpy as np
from numpy.random import choice

from symusic import Score
from miditok import REMI, TokenizerConfig
from midiutil import MIDIFile

c:\Users\GD\.conda\envs\cse253\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# You can change the random seed but try to keep your results deterministic!
# If I need to make changes to the autograder it'll require rerunning your code,
# so it should ideally generate the same results each time.
random.seed(42)

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    drive = None


### Load music dataset
We will use a subset of the [PDMX dataset](https://zenodo.org/records/14984509).

Please find the file `PDMX_subset.zip` in the homework spec.

All pieces are monophonic music (i.e. one melody line) in 4/4 time signature.

In [5]:
midi_files = glob('PDMX_subset/*.mid')
if len(midi_files) == 0:
    midi_files = glob('PDMX_subset/PDMX_subset/*.mid')
len(midi_files)


1000

### Train a tokenizer with the REMI method in MidiTok

In [6]:
config = TokenizerConfig(num_velocities=1, use_chords=False, use_programs=False)
tokenizer = REMI(config)
tokenizer.train(vocab_size=1000, files_paths=midi_files)

### Use the trained tokenizer to get tokens for each midi file
In REMI representation, each note will be represented with four tokens: `Position, Pitch, Velocity, Duration`, e.g. `('Position_28', 'Pitch_74', 'Velocity_127', 'Duration_0.4.8')`; a `Bar_None` token indicates the beginning of a new bar.

In [7]:
# e.g.:
midi = Score(midi_files[0])
tokens = tokenizer(midi)[0].tokens
tokens[:10]

['Bar_None',
 'Position_0',
 'Pitch_66',
 'Velocity_127',
 'Duration_0.4.8',
 'Position_4',
 'Pitch_62',
 'Velocity_127',
 'Duration_0.4.8',
 'Position_8']

1. Write a function to extract note pitch events from a midi file; and another extract all note pitch events from the dataset and output a dictionary that maps note pitch events to the number of times they occur in the files. (e.g. {60: 120, 61: 58, …}).

`note_extraction()`
- **Input**: a midi file

- **Output**: a list of note pitch events (e.g. [60, 62, 61, ...])

`note_frequency()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to the number of times they occur, e.g {60: 120, 61: 58, …}

In [8]:
def note_extraction(midi_file):
    # Q1a: Your code goes here
    midi = Score(midi_file)
    tokens = tokenizer(midi)[0].tokens
    return [int(token.split('_')[1]) for token in tokens if token.startswith('Pitch_')]


In [9]:
def note_frequency(midi_files):
    # Q1b: Your code goes here
    note_counts = defaultdict(int)
    for midi_file in midi_files:
        for note in note_extraction(midi_file):
            note_counts[note] += 1
    return dict(note_counts)


2. Write a function to normalize the above dictionary to produce probability scores (e.g. {60: 0.13, 61: 0.065, …})

`note_unigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: a dictionary that maps note pitch events to probabilities, e.g. {60: 0.13, 61: 0.06, …}

In [10]:
def note_unigram_probability(midi_files):
    note_counts = note_frequency(midi_files)
    unigramProbabilities = {}

    # Q2: Your code goes here
    # ...
    total = sum(note_counts.values())
    if total == 0:
        return unigramProbabilities

    for note, count in note_counts.items():
        unigramProbabilities[note] = count / total

    return unigramProbabilities


3. Generate a table of pairwise probabilities containing p(next_note | previous_note) values for the dataset; write a function that randomly generates the next note based on the previous note based on this distribution.

`note_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramTransitions`: key: previous_note, value: a list of next_note, e.g. {60:[62, 64, ..], 62:[60, 64, ..], ...} (i.e., this is a list of every other note that occured after note 60, every note that occured after note 62, etc.)

  - `bigramTransitionProbabilities`: key:previous_note, value: a list of probabilities for next_note in the same order of `bigramTransitions`, e.g. {60:[0.3, 0.4, ..], 62:[0.2, 0.1, ..], ...} (i.e., you are converting the values above to probabilities)

`sample_next_note()`
- **Input**: a note

- **Output**: next note sampled from pairwise probabilities

In [11]:
def note_bigram_probability(midi_files):
    bigramTransitions = defaultdict(list)
    bigramTransitionProbabilities = defaultdict(list)

    # Q3a: Your code goes here
    # ...
    bigramCounts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for previous_note, next_note in zip(notes[:-1], notes[1:]):
            bigramCounts[previous_note][next_note] += 1

    for previous_note, next_counts in bigramCounts.items():
        total = sum(next_counts.values())
        for next_note, count in next_counts.items():
            bigramTransitions[previous_note].append(next_note)
            bigramTransitionProbabilities[previous_note].append(count / total)

    return bigramTransitions, bigramTransitionProbabilities


In [12]:
def sample_next_note(note):
    # Q3b: Your code goes here
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    if note in bigramTransitions and len(bigramTransitions[note]) > 0:
        return random.choices(bigramTransitions[note], weights=bigramTransitionProbabilities[note])[0]

    unigramProbabilities = note_unigram_probability(midi_files)
    return random.choices(list(unigramProbabilities.keys()), weights=list(unigramProbabilities.values()))[0]


4. Write a function to calculate the perplexity of your model on a midi file.

    The perplexity of a model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-1})))$

    where $p(w_1|w_0) = p(w_1)$, $p(w_i|w_{i-1}) (i>1)$ refers to the pairwise probability p(next_note | previous_note).

`note_bigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [13]:
def note_bigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)

    # Q4: Your code goes here
    # Can use regular numpy.log (i.e., natural logarithm)
    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return np.inf

    log_probability = np.log(unigramProbabilities.get(notes[0], 1e-12))
    for previous_note, next_note in zip(notes[:-1], notes[1:]):
        probability = 1e-12
        if previous_note in bigramTransitions and next_note in bigramTransitions[previous_note]:
            index = bigramTransitions[previous_note].index(next_note)
            probability = bigramTransitionProbabilities[previous_note][index]
        log_probability += np.log(probability)

    return np.exp(-log_probability / len(notes))


5. Implement a second-order Markov chain, i.e., one which estimates p(next_note | next_previous_note, previous_note); write a function to compute the perplexity of this new model on a midi file.

    The perplexity of this model is defined as

    $\quad \text{exp}(-\frac{1}{N} \sum_{i=1}^N \text{log}(p(w_i|w_{i-2}, w_{i-1})))$

    where $p(w_1|w_{-1}, w_0) = p(w_1)$, $p(w_2|w_0, w_1) = p(w_2|w_1)$, $p(w_i|w_{i-2}, w_{i-1}) (i>2)$ refers to the probability p(next_note | next_previous_note, previous_note).


`note_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramTransitions`: key - (next_previous_note, previous_note), value - a list of next_note, e.g. {(60, 62):[64, 66, ..], (60, 64):[60, 64, ..], ...}

  - `trigramTransitionProbabilities`: key: (next_previous_note, previous_note), value: a list of probabilities for next_note in the same order of `trigramTransitions`, e.g. {(60, 62):[0.2, 0.2, ..], (60, 64):[0.4, 0.1, ..], ...}

`note_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [14]:
def note_trigram_probability(midi_files):
    trigramTransitions = defaultdict(list)
    trigramTransitionProbabilities = defaultdict(list)

    # Q5a: Your code goes here
    # ...
    trigramCounts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        notes = note_extraction(midi_file)
        for next_previous_note, previous_note, next_note in zip(notes[:-2], notes[1:-1], notes[2:]):
            trigramCounts[(next_previous_note, previous_note)][next_note] += 1

    for previous_notes, next_counts in trigramCounts.items():
        total = sum(next_counts.values())
        for next_note, count in next_counts.items():
            trigramTransitions[previous_notes].append(next_note)
            trigramTransitionProbabilities[previous_notes].append(count / total)

    return trigramTransitions, trigramTransitionProbabilities


In [15]:
def note_trigram_perplexity(midi_file):
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    # Q5b: Your code goes here
    notes = note_extraction(midi_file)
    if len(notes) == 0:
        return np.inf

    log_probability = np.log(unigramProbabilities.get(notes[0], 1e-12))
    if len(notes) > 1:
        probability = 1e-12
        if notes[0] in bigramTransitions and notes[1] in bigramTransitions[notes[0]]:
            index = bigramTransitions[notes[0]].index(notes[1])
            probability = bigramTransitionProbabilities[notes[0]][index]
        log_probability += np.log(probability)

    for next_previous_note, previous_note, next_note in zip(notes[:-2], notes[1:-1], notes[2:]):
        probability = 1e-12
        key = (next_previous_note, previous_note)
        if key in trigramTransitions and next_note in trigramTransitions[key]:
            index = trigramTransitions[key].index(next_note)
            probability = trigramTransitionProbabilities[key][index]
        log_probability += np.log(probability)

    return np.exp(-log_probability / len(notes))


6. Our model currently doesn’t have any knowledge of beats. Write a function that extracts beat lengths and outputs a list of [(beat position; beat length)] values.

    Recall that each note will be encoded as `Position, Pitch, Velocity, Duration` using REMI. Please keep the `Position` value for beat position, and convert `Duration` to beat length using provided lookup table `duration2length` (see below).

    For example, for a note represented by four tokens `('Position_24', 'Pitch_72', 'Velocity_127', 'Duration_0.4.8')`, the extracted (beat position; beat length) value is `(24, 4)`.

    As a result, we will obtain a list like [(0,8),(8,16),(24,4),(28,4),(0,4)...], where the next beat position is the previous beat position + the beat length. As we divide each bar into 32 positions by default, when reaching the end of a bar (i.e. 28 + 4 = 32 in the case of (28, 4)), the beat position reset to 0.

In [16]:
duration2length = {
    '0.2.8': 2,  # sixteenth note, 0.25 beat in 4/4 time signature
    '0.4.8': 4,  # eighth note, 0.5 beat in 4/4 time signature
    '1.0.8': 8,  # quarter note, 1 beat in 4/4 time signature
    '2.0.8': 16, # half note, 2 beats in 4/4 time signature
    '4.0.4': 32, # whole note, 4 beats in 4/4 time signature
}

`beat_extraction()`
- **Input**: a midi file

- **Output**: a list of (beat position; beat length) values

In [17]:
def beat_extraction(midi_file):
    # Q6: Your code goes here
    midi = Score(midi_file)
    tokens = tokenizer(midi)[0].tokens
    beats = []
    current_position = None

    for token in tokens:
        if token.startswith('Position_'):
            current_position = int(token.split('_')[1])
        elif token.startswith('Duration_') and current_position is not None:
            duration = token.split('_')[1]
            if duration in duration2length:
                beats.append((current_position, duration2length[duration]))
            current_position = None

    return beats


7. Implement a Markov chain that computes p(beat_length | previous_beat_length) based on the above function.

`beat_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatTransitions`: key: previous_beat_length, value: a list of beat_length, e.g. {4:[8, 2, ..], 8:[8, 4, ..], ...}

  - `bigramBeatTransitionProbabilities`: key - previous_beat_length, value - a list of probabilities for beat_length in the same order of `bigramBeatTransitions`, e.g. {4:[0.3, 0.2, ..], 8:[0.4, 0.4, ..], ...}

In [18]:
def beat_bigram_probability(midi_files):
    bigramBeatTransitions = defaultdict(list)
    bigramBeatTransitionProbabilities = defaultdict(list)

    # Q7: Your code goes here
    # ...
    bigramBeatCounts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        beat_lengths = [beat_length for _, beat_length in beat_extraction(midi_file)]
        for previous_beat_length, beat_length in zip(beat_lengths[:-1], beat_lengths[1:]):
            bigramBeatCounts[previous_beat_length][beat_length] += 1

    for previous_beat_length, beat_counts in bigramBeatCounts.items():
        total = sum(beat_counts.values())
        for beat_length, count in beat_counts.items():
            bigramBeatTransitions[previous_beat_length].append(beat_length)
            bigramBeatTransitionProbabilities[previous_beat_length].append(count / total)

    return bigramBeatTransitions, bigramBeatTransitionProbabilities


8. Implement a function to compute p(beat length | beat position), and compute the perplexity of your models from Q7 and Q8. For both models, we only consider the probabilities of predicting the sequence of **beat lengths**.

`beat_pos_bigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `bigramBeatPosTransitions`: key - beat_position, value - a list of beat_length

  - `bigramBeatPosTransitionProbabilities`: key - beat_position, value - a list of probabilities for beat_length in the same order of `bigramBeatPosTransitions`

`beat_bigram_perplexity()`
- **Input**: a midi file

- **Output**: two perplexity values correspond to the models in Q7 and Q8, respectively

In [19]:
def beat_pos_bigram_probability(midi_files):
    bigramBeatPosTransitions = defaultdict(list)
    bigramBeatPosTransitionProbabilities = defaultdict(list)

    # Q8a: Your code goes here
    # ...
    bigramBeatPosCounts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        for beat_position, beat_length in beat_extraction(midi_file):
            bigramBeatPosCounts[beat_position][beat_length] += 1

    for beat_position, beat_counts in bigramBeatPosCounts.items():
        total = sum(beat_counts.values())
        for beat_length, count in beat_counts.items():
            bigramBeatPosTransitions[beat_position].append(beat_length)
            bigramBeatPosTransitionProbabilities[beat_position].append(count / total)

    return bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities


In [20]:
def beat_bigram_perplexity(midi_file):
    bigramBeatTransitions, bigramBeatTransitionProbabilities = beat_bigram_probability(midi_files)
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    # Q8b: Your code goes here
    # Hint: one more probability function needs to be computed
    beat_counts = defaultdict(int)
    for train_file in midi_files:
        for _, beat_length in beat_extraction(train_file):
            beat_counts[beat_length] += 1
    total_beats = sum(beat_counts.values())
    beat_unigram_probabilities = {beat_length: count / total_beats for beat_length, count in beat_counts.items()}

    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return np.inf, np.inf

    # perplexity for Q7
    log_probability_Q7 = np.log(beat_unigram_probabilities.get(beats[0][1], 1e-12))
    for (_, previous_beat_length), (_, beat_length) in zip(beats[:-1], beats[1:]):
        probability = 1e-12
        if previous_beat_length in bigramBeatTransitions and beat_length in bigramBeatTransitions[previous_beat_length]:
            index = bigramBeatTransitions[previous_beat_length].index(beat_length)
            probability = bigramBeatTransitionProbabilities[previous_beat_length][index]
        log_probability_Q7 += np.log(probability)
    perplexity_Q7 = np.exp(-log_probability_Q7 / len(beats))

    # perplexity for Q8
    log_probability_Q8 = 0
    for beat_position, beat_length in beats:
        probability = 1e-12
        if beat_position in bigramBeatPosTransitions and beat_length in bigramBeatPosTransitions[beat_position]:
            index = bigramBeatPosTransitions[beat_position].index(beat_length)
            probability = bigramBeatPosTransitionProbabilities[beat_position][index]
        log_probability_Q8 += np.log(probability)
    perplexity_Q8 = np.exp(-log_probability_Q8 / len(beats))

    return perplexity_Q7, perplexity_Q8


9. Implement a Markov chain that computes p(beat_length | previous_beat_length, beat_position), and report its perplexity.

`beat_trigram_probability()`
- **Input**: all midi files `midi_files`

- **Output**: two dictionaries:

  - `trigramBeatTransitions`: key: (previous_beat_length, beat_position), value: a list of beat_length

  - `trigramBeatTransitionProbabilities`: key: (previous_beat_length, beat_position), value: a list of probabilities for beat_length in the same order of `trigramBeatTransitions`

`beat_trigram_perplexity()`
- **Input**: a midi file

- **Output**: perplexity value

In [21]:
def beat_trigram_probability(midi_files):
    trigramBeatTransitions = defaultdict(list)
    trigramBeatTransitionProbabilities = defaultdict(list)

    # Q9a: Your code goes here
    # ...
    trigramBeatCounts = defaultdict(lambda: defaultdict(int))
    for midi_file in midi_files:
        beats = beat_extraction(midi_file)
        for (_, previous_beat_length), (beat_position, beat_length) in zip(beats[:-1], beats[1:]):
            trigramBeatCounts[(previous_beat_length, beat_position)][beat_length] += 1

    for previous_context, beat_counts in trigramBeatCounts.items():
        total = sum(beat_counts.values())
        for beat_length, count in beat_counts.items():
            trigramBeatTransitions[previous_context].append(beat_length)
            trigramBeatTransitionProbabilities[previous_context].append(count / total)

    return trigramBeatTransitions, trigramBeatTransitionProbabilities


In [22]:
def beat_trigram_perplexity(midi_file):
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    trigramBeatTransitions, trigramBeatTransitionProbabilities = beat_trigram_probability(midi_files)
    # Q9b: Your code goes here
    beats = beat_extraction(midi_file)
    if len(beats) == 0:
        return np.inf

    first_position, first_length = beats[0]
    first_probability = 1e-12
    if first_position in bigramBeatPosTransitions and first_length in bigramBeatPosTransitions[first_position]:
        index = bigramBeatPosTransitions[first_position].index(first_length)
        first_probability = bigramBeatPosTransitionProbabilities[first_position][index]
    log_probability = np.log(first_probability)

    for (_, previous_beat_length), (beat_position, beat_length) in zip(beats[:-1], beats[1:]):
        probability = 1e-12
        key = (previous_beat_length, beat_position)
        if key in trigramBeatTransitions and beat_length in trigramBeatTransitions[key]:
            index = trigramBeatTransitions[key].index(beat_length)
            probability = trigramBeatTransitionProbabilities[key][index]
        log_probability += np.log(probability)

    return np.exp(-log_probability / len(beats))


10. Use the model from Q5 to generate N notes, and the model from Q8 to generate beat lengths for each note. Save the generated music as a midi file (see code from workbook1) as q10.mid. Remember to reset the beat position to 0 when reaching the end of a bar.

`music_generate`
- **Input**: target length, e.g. 500

- **Output**: a midi file q10.mid

Note: the duration of one beat in MIDIUtil is 1, while in MidiTok is 8. Divide beat length by 8 if you use methods in MIDIUtil to save midi files.

In [23]:
def music_generate(length):
    # sample notes
    unigramProbabilities = note_unigram_probability(midi_files)
    bigramTransitions, bigramTransitionProbabilities = note_bigram_probability(midi_files)
    trigramTransitions, trigramTransitionProbabilities = note_trigram_probability(midi_files)

    # Q10: Your code goes here ...
    sampled_notes = []
    if length > 0:
        sampled_notes.append(random.choices(list(unigramProbabilities.keys()), weights=list(unigramProbabilities.values()))[0])
    if length > 1:
        previous_note = sampled_notes[-1]
        if previous_note in bigramTransitions:
            sampled_notes.append(random.choices(bigramTransitions[previous_note], weights=bigramTransitionProbabilities[previous_note])[0])
        else:
            sampled_notes.append(random.choices(list(unigramProbabilities.keys()), weights=list(unigramProbabilities.values()))[0])

    while len(sampled_notes) < length:
        key = (sampled_notes[-2], sampled_notes[-1])
        if key in trigramTransitions:
            next_note = random.choices(trigramTransitions[key], weights=trigramTransitionProbabilities[key])[0]
        elif sampled_notes[-1] in bigramTransitions:
            next_note = random.choices(bigramTransitions[sampled_notes[-1]], weights=bigramTransitionProbabilities[sampled_notes[-1]])[0]
        else:
            next_note = random.choices(list(unigramProbabilities.keys()), weights=list(unigramProbabilities.values()))[0]
        sampled_notes.append(next_note)

    # sample beats
    bigramBeatPosTransitions, bigramBeatPosTransitionProbabilities = beat_pos_bigram_probability(midi_files)
    sampled_beats = []
    beat_position = 0
    for _ in range(length):
        if beat_position in bigramBeatPosTransitions:
            beat_length = random.choices(
                bigramBeatPosTransitions[beat_position],
                weights=bigramBeatPosTransitionProbabilities[beat_position]
            )[0]
        else:
            beat_length = random.choice(list(duration2length.values()))
        sampled_beats.append((beat_position, beat_length))
        beat_position = (beat_position + beat_length) % 32

    # save the generated music as a midi file
    midi = MIDIFile(1)
    track = 0
    channel = 0
    time = 0
    tempo = 120
    volume = 100
    midi.addTempo(track, time, tempo)

    for note, (_, beat_length) in zip(sampled_notes, sampled_beats):
        duration = beat_length / 8
        midi.addNote(track, channel, note, time, duration, volume)
        time += duration

    with open('q10.mid', 'wb') as output_file:
        midi.writeFile(output_file)
